# 🍔 AestheticRankNet — Training Notebook
โมเดล: `convnext_base_w` | `ViT-B-16`

## 0. Import Libraries

In [1]:
import os
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

## 1. Pipeline Configuration ⚙️

In [2]:
class TrainingConfig:
    seed       = 42
    num_epochs = 15
    num_folds  = 5

    csv_questionnaire = r'Dataset_for_development\data_from_questionaire.csv'
    csv_instagram     = r'Dataset_for_development\data_from_intragram.csv'
    img_dirs = [
        r'G:\MachineVision\Dataset_for_development\Questionair Images',
        r'G:\MachineVision\Dataset_for_development\Instagram Photos',
    ]

    # รัน 2 ตัวนี้ต่อกันเลย
    # 🟢 เปลี่ยนจากแบบเดิมเป็นแบบนี้ครับ
    models_to_train = ['convnext_base_w', 'convnext_large_d']

    mean = [0.481, 0.457, 0.408]
    std  = [0.268, 0.261, 0.275]

## 2. Utility & Loss Functions 🛠️

In [3]:
import torch.nn.functional as F

def set_deterministic_mode(seed: int = 42) -> None:
    """กำหนดค่า Random Seed เพื่อควบคุมการทดลองให้ได้ผลลัพธ์คงที่"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f"[System] กำหนดค่า Random Seed: {seed}")


def compute_weighted_bce(diff: torch.Tensor, target: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
    """ใช้ F.binary_cross_entropy_with_logits ป้องกัน ViT-B-16 คำนวณค่า Sigmoid ระเบิด"""
    # 🟢 บังคับแปลงเป็น Float32 ก่อนคิด Loss เสมอ เพื่อความแม่นยำระดับสูงสุด
    loss = F.binary_cross_entropy_with_logits(diff.float(), target.float(), reduction='none')
    return (loss * weight).mean()


def serialize_model(model: nn.Module, save_path: str, max_retries: int = 3) -> bool:
    """บันทึกพารามิเตอร์ของโมเดล พร้อมระบบทำงานซ้ำกรณีเกิดข้อผิดพลาดด้าน I/O"""
    for attempt in range(max_retries):
        try:
            torch.save(model.state_dict(), save_path)
            print(f"   [Save] บันทึก Checkpoint สำเร็จ: {save_path}")
            return True
        except Exception as e:
            print(f"   [Warning] ไม่สามารถเขียนไฟล์ได้ในความพยายามครั้งที่ {attempt+1}: {e}")
            time.sleep(1)
    print("   [Error] ล้มเหลวในการบันทึกโมเดล โปรดตรวจสอบสิทธิ์การเข้าถึงพื้นที่จัดเก็บข้อมูล")
    return False

## 3. Data Processing 🗂️

In [4]:
def initialize_dataframes() -> pd.DataFrame:
    """เตรียมข้อมูลจากไฟล์ CSV และคำนวณ Soft Label"""
    df_q  = pd.read_csv(TrainingConfig.csv_questionnaire)
    df_ig = pd.read_csv(TrainingConfig.csv_instagram)

    df_q['Soft_Label'] = df_q['Num Vote 2'] / (df_q['Num Vote 1'] + df_q['Num Vote 2'])
    df_q['Weight']     = df_q['Num Voter']  / df_q['Num Voter'].max()

    swap_idx = df_ig.sample(frac=0.5, random_state=TrainingConfig.seed).index
    temp = df_ig.loc[swap_idx, 'Image 1'].copy()
    df_ig.loc[swap_idx, 'Image 1'] = df_ig.loc[swap_idx, 'Image 2']
    df_ig.loc[swap_idx, 'Image 2'] = temp
    df_ig.loc[swap_idx, 'Winner']  = 2

    df_ig['Soft_Label'] = df_ig['Winner'].apply(lambda x: 0.9 if x == 2 else 0.1)
    df_ig['Weight']     = 0.5

    return pd.concat([df_q, df_ig], ignore_index=True)


class VisualPairDataset(Dataset):
    """คลาสสำหรับจัดการคู่รูปภาพและดึงข้อมูลเข้าสู่ DataLoader"""
    def __init__(self, df: pd.DataFrame, img_dirs: list, transform=None):
        self.transform  = transform
        self.img_paths: dict[str, str] = {}

        for d in img_dirs:
            if os.path.exists(d):
                for root, _, files in os.walk(d):
                    for f in files:
                        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                            self.img_paths[f.lower()] = os.path.join(root, f)

        valid_rows = []
        for _, row in df.iterrows():
            i1 = str(row['Image 1']).strip().lower()
            i2 = str(row['Image 2']).strip().lower()
            if i1 in self.img_paths and i2 in self.img_paths:
                row['Image 1'], row['Image 2'] = i1, i2
                valid_rows.append(row)
        self.df = pd.DataFrame(valid_rows).reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row  = self.df.iloc[idx]
        img1 = Image.open(self.img_paths[row['Image 1']]).convert('RGB')
        img2 = Image.open(self.img_paths[row['Image 2']]).convert('RGB')
        if self.transform:
            img1, img2 = self.transform(img1), self.transform(img2)
        soft_lbl = torch.tensor(row['Soft_Label'], dtype=torch.float32)
        weight   = torch.tensor(row['Weight'],     dtype=torch.float32)
        return img1, img2, soft_lbl, weight

## 4. Model Architecture 🧠

In [5]:
import torch.nn.functional as F
class AestheticScorer(nn.Module):
    """โมเดลประเมินความสวยงามของภาพ ทำงานร่วมกับ CLIP Backbone"""
    def __init__(self, model_type: str = 'ViT-B-16'):
        super().__init__()
        self.model_type = model_type

        config_pretrained = {
            # 🟢 แก้ไขบรรทัดนี้เป็น s26b_b102k แทนครับ
            'convnext_base_w':  'laion2b_s13b_b82k_augreg',
            'convnext_large_d': 'laion2b_s26b_b102k_augreg',

        }
        pretrained = config_pretrained.get(model_type, 'openai')
        self.backbone, _, _ = open_clip.create_model_and_transforms(model_type, pretrained=pretrained)

        # ตรวจสอบขนาด Feature Dimension อัตโนมัติ
        if hasattr(self.backbone.visual, 'output_dim'):
            feat_dim = self.backbone.visual.output_dim
        elif hasattr(self.backbone.visual, 'num_features'):
            feat_dim = self.backbone.visual.num_features
        else:
            with torch.no_grad():
                dummy_input = torch.zeros(1, 3, 224, 224)
                feat_dim = self.backbone.encode_image(dummy_input).shape[1]

        print(f"[Init] สถาปัตยกรรม {model_type} | Feature Dimension: {feat_dim}")

        self.scorer_head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, img: torch.Tensor) -> torch.Tensor:
        # 1. สกัด Feature
        features = self.backbone.encode_image(img)
        
        # 2. 🟢 ใช้ Bfloat16 รันยาวๆ (ไม่ต้องแปลงเป็น Float32 แล้ว)
        features = torch.nn.functional.normalize(features, p=2, dim=-1)
        
        # 3. ส่งให้ Head
        return self.scorer_head(features).view(-1)

    def apply_partial_freeze(self) -> None:
        """ระงับการอัปเดตน้ำหนักส่วนใหญ่ของ Backbone (Freeze) และเปิดเฉพาะส่วนท้าย"""
        for param in self.parameters():
            param.requires_grad = False
        for param in self.scorer_head.parameters():
            param.requires_grad = True
        visual_params = list(self.backbone.visual.parameters())
        num_unfreeze  = int(len(visual_params) * 0.2)
        for param in visual_params[-num_unfreeze:]:
            param.requires_grad = True

## 5. Training Engine 🚀

In [6]:
def execute_train_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    device: torch.device,
    epoch: int,
) -> float:
    """ฟังก์ชันควบคุมการรัน Forward และ Backward Pass ต่อ 1 Epoch"""
    model.train()
    running_loss = 0.0
    pbar = tqdm(dataloader, desc=f"   [Train] Epoch {epoch}/{TrainingConfig.num_epochs}")

    for i1, i2, target, w in pbar:
        # สุ่มสลับตำแหน่งภาพ (Data Augmentation)
        if random.random() > 0.5:
            i1, i2, target = i2.clone(), i1.clone(), 1.0 - target

        i1, i2, target, w = i1.to(device), i2.to(device), target.to(device), w.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            loss = compute_weighted_bce(model(i2) - model(i1), target, w)   

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    return running_loss / len(dataloader)


def execute_validation(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
) -> float:
    """ฟังก์ชันทดสอบความแม่นยำของโมเดล (Validation)"""
    model.eval()
    all_probs, all_targets = [], []

    with torch.no_grad():
        for i1, i2, target, _ in dataloader:
            i1, i2 = i1.to(device), i2.to(device)
            
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                # ใช้เทคนิค Test Time Augmentation (TTA) โดยประเมินร่วมกับภาพกลับด้าน
                diff = (model(i2) - model(i1) + model(torch.flip(i2, [3])) - model(torch.flip(i1, [3]))) / 2.0
            # 🟢 เติม .float() เข้าไปตรงนี้ เพื่อแปลง Bfloat16 เป็น Float32 ให้ NumPy อ่านออก
            all_probs.extend(torch.sigmoid(diff).float().cpu().numpy())
            all_targets.extend((target >= 0.5).int().numpy())

    return roc_auc_score(all_targets, all_probs)

## 6. Setup & Data Preparation

In [7]:
set_deterministic_mode(TrainingConfig.seed)

device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
active_gpu = torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'
print(f"[System] ตรวจพบอุปกรณ์ประมวลผล: {active_gpu}\n")

df_all      = initialize_dataframes()
unique_imgs = list(set(df_all['Image 1'].str.lower()).union(set(df_all['Image 2'].str.lower())))
print(f"[Data] โหลดข้อมูลสำเร็จ: {len(df_all)} คู่ | ภาพ unique: {len(unique_imgs)} ไฟล์")
df_all.head()

[System] กำหนดค่า Random Seed: 42
[System] ตรวจพบอุปกรณ์ประมวลผล: NVIDIA GeForce RTX 4060 Laptop GPU

[Data] โหลดข้อมูลสำเร็จ: 765 คู่ | ภาพ unique: 1493 ไฟล์


,Image 1,Image 2,Menu,Winner,Num Voter,Num Vote 1,Num Vote 2,Soft_Label,Weight
0,s1_1.jpg,s1_2.jpg,Sushi,1,129.0,81.0,48.0,0.372093,1.0
1,s2_1.jpg,s2_2.jpg,Sushi,1,129.0,75.0,54.0,0.418605,1.0
2,s3_1.jpg,s3_2.jpg,Sushi,1,129.0,84.0,45.0,0.348837,1.0
3,s4_1.jpg,s4_2.jpg,Sushi,1,129.0,115.0,14.0,0.108527,1.0
4,s5_1.jpg,s5_2.jpg,Sushi,1,129.0,101.0,28.0,0.217054,1.0


## 7. Main Training Loop 🏁

In [8]:
# ==========================================
# 🟢 CELL 1: PHASE 1 (Cross Validation 5 Folds)
# ==========================================
import torch
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from torchvision import transforms

# ตัวแปรนี้จะจำไว้ว่าโมเดลไหน Fold ไหนดีที่สุด เพื่อส่งต่อให้ Phase 2
phase1_results = {} 

print(f"{'='*50}\n🚀 เริ่มต้นเทรน PHASE 1\n{'='*50}")

for m_type in TrainingConfig.models_to_train:
    print(f"\n{'='*50}\n[Process] เริ่มต้นเทรนโมเดล: {m_type}\n{'='*50}")

    # ปรับ Batch Size อัตโนมัติ ป้องกัน VRAM ระเบิดตอนรันตัวใหญ่
    batch_size = 4 if m_type == 'convnext_large_d' else 8
    img_size = 224

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(TrainingConfig.mean, TrainingConfig.std),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(TrainingConfig.mean, TrainingConfig.std),
    ])

    kf = KFold(n_splits=TrainingConfig.num_folds, shuffle=True, random_state=TrainingConfig.seed)
    
    model_best_auc = 0.0
    model_best_fold = 1

    for fold, (train_img_idx, val_img_idx) in enumerate(kf.split(unique_imgs)):
        val_imgs_set = set(np.array(unique_imgs)[val_img_idx])
        val_mask     = (
            df_all['Image 1'].str.lower().isin(val_imgs_set) |
            df_all['Image 2'].str.lower().isin(val_imgs_set)
        )
        train_df = df_all[~val_mask].reset_index(drop=True)
        val_df   = df_all[ val_mask].reset_index(drop=True)

        if (val_df['Soft_Label'] >= 0.5).astype(int).nunique() < 2:
            continue

        print(f"\n[Fold] {m_type} — Fold {fold+1}/{TrainingConfig.num_folds} | Train: {len(train_df)} คู่ | Val: {len(val_df)} คู่")

        loader_tr  = DataLoader(VisualPairDataset(train_df, TrainingConfig.img_dirs, train_tf), batch_size=batch_size, shuffle=True)
        loader_val = DataLoader(VisualPairDataset(val_df,   TrainingConfig.img_dirs, val_tf),   batch_size=batch_size, shuffle=False)

        model = AestheticScorer(model_type=m_type).to(device)
        model.apply_partial_freeze()

        optimizer = optim.AdamW([
            {'params': [p for n, p in model.named_parameters() if p.requires_grad and 'scorer' not in n], 'lr': 1e-6},
            {'params': [p for n, p in model.named_parameters() if p.requires_grad and 'scorer'     in n], 'lr': 1e-4},
        ], weight_decay=1e-4)
        
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TrainingConfig.num_epochs)
        scaler    = torch.amp.GradScaler('cuda')

        best_auc = 0.0
        for epoch in range(1, TrainingConfig.num_epochs + 1):
            train_loss = execute_train_epoch(model, loader_tr, optimizer, scaler, device, epoch)
            scheduler.step()

            val_auc = execute_validation(model, loader_val, device)
            print(f"   [Result] Epoch {epoch:>2} | Train Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f}")

            if val_auc > best_auc:
                best_auc  = val_auc
                save_name = f'AestheticRankNet_{m_type}_Fold{fold+1}.pth'
                serialize_model(model, save_name)

        print(f"\n[Done] Fold {fold+1} สิ้นสุด — Best AUC: {best_auc:.4f}")
        
        # ค้นหาว่า Fold ไหนดีที่สุดของโมเดลนี้
        if best_auc > model_best_auc:
            model_best_auc = best_auc
            model_best_fold = fold + 1
            
        del model
        torch.cuda.empty_cache()

    # บันทึกสถิติที่ดีที่สุดของโมเดลนี้ไว้เตรียมส่งให้ Phase 2
    phase1_results[m_type] = {'best_fold': model_best_fold, 'auc': model_best_auc}
    print(f"\n🏆 สรุป Phase 1 ของ {m_type} -> Fold ที่ดีที่สุดคือ {model_best_fold} (AUC: {model_best_auc:.4f})")

print("\n[System] สิ้นสุดกระบวนการเทรน Phase 1 อย่างสมบูรณ์! พร้อมรัน Cell ที่ 2 ได้เลย")

🚀 เริ่มต้นเทรน PHASE 1

[Process] เริ่มต้นเทรนโมเดล: convnext_base_w

[Fold] convnext_base_w — Fold 1/5 | Train: 491 คู่ | Val: 274 คู่
[Init] สถาปัตยกรรม convnext_base_w | Feature Dimension: 640


   [Train] Epoch 1/15:   0%|          | 0/62 [00:00<?, ?it/s]

   [Result] Epoch  1 | Train Loss: 0.5600 | Val AUC: 0.6651
   [Save] บันทึก Checkpoint สำเร็จ: AestheticRankNet_convnext_base_w_Fold1.pth


   [Train] Epoch 2/15:   0%|          | 0/62 [00:00<?, ?it/s]

   [Result] Epoch  2 | Train Loss: 0.5373 | Val AUC: 0.7320
   [Save] บันทึก Checkpoint สำเร็จ: AestheticRankNet_convnext_base_w_Fold1.pth


...

   [Result] Epoch 13 | Train Loss: 0.4622 | Val AUC: 0.8182
   [Save] บันทึก Checkpoint สำเร็จ: AestheticRankNet_convnext_large_d_Fold5.pth


   [Train] Epoch 14/15:   0%|          | 0/122 [00:00<?, ?it/s]

   [Result] Epoch 14 | Train Loss: 0.4645 | Val AUC: 0.8191
   [Save] บันทึก Checkpoint สำเร็จ: AestheticRankNet_convnext_large_d_Fold5.pth


   [Train] Epoch 15/15:   0%|          | 0/122 [00:00<?, ?it/s]

   [Result] Epoch 15 | Train Loss: 0.4621 | Val AUC: 0.8188

[Done] Fold 5 สิ้นสุด — Best AUC: 0.8191

🏆 สรุป Phase 1 ของ convnext_large_d -> Fold ที่ดีที่สุดคือ 5 (AUC: 0.8191)

[System] สิ้นสุดกระบวนการเทรน Phase 1 อย่างสมบูรณ์! พร้อมรัน Cell ที่ 2 ได้เลย


## 7.5 เฟส2 ลดLearning rate

In [9]:
# ==========================================
# 🔵 CELL 2: PHASE 2 (Deep Fine-tuning)
# ==========================================
import torch
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold

def apply_phase2_unfreeze(model):
    """ฟังก์ชันปลดล็อกสมองระดับ 2 (50%)"""
    for param in model.parameters():
        param.requires_grad = False
    for param in model.scorer_head.parameters():
        param.requires_grad = True
    visual_params = list(model.backbone.visual.parameters())
    num_unfreeze  = int(len(visual_params) * 0.5)
    for param in visual_params[-num_unfreeze:]:
        param.requires_grad = True

print(f"{'='*50}\n🚀 เริ่มต้นเทรน PHASE 2 (ดึงพลังหยดสุดท้าย)\n{'='*50}")

# ⚠️ ป้องกันกรณี Kernel ดับ แล้วตัวแปรหาย คุณสามารถกรอกเลขด้วยตัวเองได้ตรงนี้
# phase1_results = {'convnext_base_w': {'best_fold': 1}, 'convnext_large_d': {'best_fold': 3}}

if 'phase1_results' not in locals():
    print("❌ ไม่พบข้อมูลจาก Phase 1 กรุณารัน Cell 1 ก่อน หรือระบุ phase1_results ด้วยตัวเอง")
else:
    for m_type, data in phase1_results.items():
        best_fold = data['best_fold']
        print(f"\n🌟 กำลังทำ Phase 2 ให้กับ {m_type} (ใช้ความรู้จาก Fold {best_fold})")
        
        batch_size = 4 if m_type == 'convnext_large_d' else 8
        phase2_epochs = 5
        
        # --- 1. เตรียมข้อมูลเฉพาะของ Fold ที่ดีที่สุด ---
        kf = KFold(n_splits=TrainingConfig.num_folds, shuffle=True, random_state=TrainingConfig.seed)
        for fold, (train_img_idx, val_img_idx) in enumerate(kf.split(unique_imgs)):
            if fold + 1 == best_fold:
                val_imgs_set = set(np.array(unique_imgs)[val_img_idx])
                val_mask     = (df_all['Image 1'].str.lower().isin(val_imgs_set) | df_all['Image 2'].str.lower().isin(val_imgs_set))
                
                train_df = df_all[~val_mask].reset_index(drop=True)
                val_df   = df_all[ val_mask].reset_index(drop=True)
                
                # ใช้ train_tf และ val_tf ที่เคยนิยามไว้ในเซลล์ด้านบน
                loader_tr  = DataLoader(VisualPairDataset(train_df, TrainingConfig.img_dirs, train_tf), batch_size=batch_size, shuffle=True)
                loader_val = DataLoader(VisualPairDataset(val_df,   TrainingConfig.img_dirs, val_tf),   batch_size=batch_size, shuffle=False)
                break

        # --- 2. โหลด Weight และลด Learning Rate ---
        model = AestheticScorer(model_type=m_type).to(device)
        weight_path = f'AestheticRankNet_{m_type}_Fold{best_fold}.pth'
        
        try:
            model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
            print(f"   ✅ โหลดไฟล์ {weight_path} สำเร็จ")
        except Exception as e:
            print(f"   ❌ เกิดข้อผิดพลาดในการโหลดไฟล์ {weight_path}: {e}")
            continue

        apply_phase2_unfreeze(model)
        
        optimizer = optim.AdamW([
            {'params': [p for n, p in model.named_parameters() if p.requires_grad and 'scorer' not in n], 'lr': 1e-7},
            {'params': [p for n, p in model.named_parameters() if p.requires_grad and 'scorer'     in n], 'lr': 1e-5},
        ], weight_decay=1e-4)
        
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=phase2_epochs)
        scaler    = torch.amp.GradScaler('cuda')
        
        # --- 3. รันเทรน Phase 2 ---
        best_auc_phase2 = 0.0
        for epoch in range(1, phase2_epochs + 1):
            train_loss = execute_train_epoch(model, loader_tr, optimizer, scaler, device, epoch)
            scheduler.step()

            val_auc = execute_validation(model, loader_val, device)
            print(f"   [Phase 2 - Epoch {epoch:>2}] Train Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f}")

            if val_auc > best_auc_phase2:
                best_auc_phase2 = val_auc
                save_name = f'AestheticRankNet_{m_type}_Phase2_Best.pth'
                serialize_model(model, save_name)

        print(f"\n🎉 จบ Phase 2 ของ {m_type}! | AUC สุดท้าย: {best_auc_phase2:.4f}")
        del model
        torch.cuda.empty_cache()

    print("\n[System] สิ้นสุดกระบวนการ Phase 2 ทั้งหมดอย่างสมบูรณ์!")

🚀 เริ่มต้นเทรน PHASE 2 (ดึงพลังหยดสุดท้าย)

🌟 กำลังทำ Phase 2 ให้กับ convnext_base_w (ใช้ความรู้จาก Fold 2)
[Init] สถาปัตยกรรม convnext_base_w | Feature Dimension: 640
   ✅ โหลดไฟล์ AestheticRankNet_convnext_base_w_Fold2.pth สำเร็จ


   [Train] Epoch 1/15:   0%|          | 0/62 [00:00<?, ?it/s]

   [Phase 2 - Epoch  1] Train Loss: 0.4989 | Val AUC: 0.8356
   [Save] บันทึก Checkpoint สำเร็จ: AestheticRankNet_convnext_base_w_Phase2_Best.pth


   [Train] Epoch 2/15:   0%|          | 0/62 [00:00<?, ?it/s]

   [Phase 2 - Epoch  2] Train Loss: 0.4982 | Val AUC: 0.8359
   [Save] บันทึก Checkpoint สำเร็จ: AestheticRankNet_convnext_base_w_Phase2_Best.pth


...

   [Phase 2 - Epoch  3] Train Loss: 0.4637 | Val AUC: 0.8203


   [Train] Epoch 4/15:   0%|          | 0/122 [00:00<?, ?it/s]

   [Phase 2 - Epoch  4] Train Loss: 0.4610 | Val AUC: 0.8209
   [Save] บันทึก Checkpoint สำเร็จ: AestheticRankNet_convnext_large_d_Phase2_Best.pth


   [Train] Epoch 5/15:   0%|          | 0/122 [00:00<?, ?it/s]

   [Phase 2 - Epoch  5] Train Loss: 0.4632 | Val AUC: 0.8208

🎉 จบ Phase 2 ของ convnext_large_d! | AUC สุดท้าย: 0.8209

[System] สิ้นสุดกระบวนการ Phase 2 ทั้งหมดอย่างสมบูรณ์!


## 9. Essemble model


In [10]:
# ==========================================
# 🌟 โค้ดใช้งานจริง: Ultimate Weighted Ensemble (Phase 2) 🌟
# ==========================================
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

print(f"{'='*60}\n🚀 โหลดสุดยอดโมเดลจาก Phase 2 พร้อมทำงาน!\n{'='*60}")

# 1. สร้างร่างโมเดล
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_base  = AestheticScorer(model_type='convnext_base_w').to(device)
model_large = AestheticScorer(model_type='convnext_large_d').to(device)

# 2. 🟢 โหลดไฟล์น้ำหนักจาก Phase 2
try:
    model_base.load_state_dict(torch.load('AestheticRankNet_convnext_base_w_Phase2_Best.pth', map_location=device, weights_only=True))
    model_large.load_state_dict(torch.load('AestheticRankNet_convnext_large_d_Phase2_Best.pth', map_location=device, weights_only=True))
    print("   ✅ อัญเชิญพลัง Phase 2 (Base + Large) สำเร็จ!")
except FileNotFoundError:
    print("   ❌ หาไฟล์ Phase 2 ไม่เจอ ตรวจสอบให้แน่ใจว่ารัน Phase 2 จบแล้วนะครับ")

model_base.eval()
model_large.eval()

# 3. เตรียมฟังก์ชันแปลงภาพ
img_size = 224
infer_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(TrainingConfig.mean, TrainingConfig.std),
])

def predict_better_image(img_path1: str, img_path2: str, weight_base=0.4, weight_large=0.6):
    """ฟังก์ชันตัดสินว่าภาพไหนสวยกว่ากัน ด้วยพลัง Ensemble"""
    # โหลดและแปลงรูปภาพ
    img1_pil = Image.open(img_path1).convert('RGB')
    img2_pil = Image.open(img_path2).convert('RGB')
    
    img1_tensor = infer_tf(img1_pil).unsqueeze(0).to(device) # เพิ่ม batch dimension
    img2_tensor = infer_tf(img2_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            # 🟢 ให้รุ่น Base ประเมิน (ใช้ TTA)
            diff_base = (
                model_base(img2_tensor) - model_base(img1_tensor) + 
                model_base(torch.flip(img2_tensor, [3])) - model_base(torch.flip(img1_tensor, [3]))
            ) / 2.0
            prob_base = torch.sigmoid(diff_base).float()
            
            # 🟢 ให้รุ่น Large ประเมิน (ใช้ TTA)
            diff_large = (
                model_large(img2_tensor) - model_large(img1_tensor) + 
                model_large(torch.flip(img2_tensor, [3])) - model_large(torch.flip(img1_tensor, [3]))
            ) / 2.0
            prob_large = torch.sigmoid(diff_large).float()
            
            # 🟢 รวมคะแนน (Weighted Ensemble)
            final_prob = (prob_base * weight_base) + (prob_large * weight_large)
            score = final_prob.item()

    # แสดงผลลัพธ์
    winner = "ภาพที่ 2" if score >= 0.5 else "ภาพที่ 1"
    confidence = score if score >= 0.5 else 1.0 - score
    
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img1_pil); axes[0].set_title("Image 1"); axes[0].axis('off')
    axes[1].imshow(img2_pil); axes[1].set_title("Image 2"); axes[1].axis('off')
    plt.suptitle(f"🏆 AI ตัดสินให้ '{winner}' สวยกว่า!\n(ความมั่นใจ: {confidence*100:.2f}%)", fontsize=14, fontweight='bold')
    plt.show()

print("\n[System] ฟังก์ชัน predict_better_image() พร้อมใช้งานแล้ว!")

🚀 โหลดสุดยอดโมเดลจาก Phase 2 พร้อมทำงาน!
[Init] สถาปัตยกรรม convnext_base_w | Feature Dimension: 640
[Init] สถาปัตยกรรม convnext_large_d | Feature Dimension: 768
   ✅ อัญเชิญพลัง Phase 2 (Base + Large) สำเร็จ!

[System] ฟังก์ชัน predict_better_image() พร้อมใช้งานแล้ว!


## 9 Predicter เอารูปมาเลย 🧙

In [12]:
import os
import pandas as pd
import torch
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm # ใช้ tqdm.auto จะแสดงแถบโหลดใน Notebook สวยกว่า

print(f"{'='*60}\n🚀 เริ่มกระบวนการ Predict Test Set (Weighted Ensemble)\n{'='*60}")

# ── 1. ตั้งค่า Path (อ้างอิงจากโค้ดเดิมของคุณ) ───────────────────────
CSV_IN   = r'G:\MachineVision\Test Set 1_268\Test Set 1\test.csv'
CSV_OUT  = r'G:\MachineVision\Test Set 1_268\Test Set 1\test_predict.csv'
IMG_DIR  = r'G:\MachineVision\Test Set 1_268\Test Set 1\Test Images'

# ── 2. เตรียมโมเดลและเครื่องมือ ──────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# สร้างฟังก์ชันแปลงภาพ
mean = [0.481, 0.457, 0.408]
std  = [0.268, 0.261, 0.275]
infer_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# อัญเชิญสุดยอดโมเดล Phase 2 
model_base  = AestheticScorer(model_type='convnext_base_w').to(device)
model_large = AestheticScorer(model_type='convnext_large_d').to(device)

try:
    model_base.load_state_dict(torch.load('AestheticRankNet_convnext_base_w_Phase2_Best.pth', map_location=device, weights_only=True))
    model_large.load_state_dict(torch.load('AestheticRankNet_convnext_large_d_Phase2_Best.pth', map_location=device, weights_only=True))
    print("✅ โหลดไฟล์น้ำหนัก Phase 2 (Base + Large) สำเร็จ!")
except FileNotFoundError:
    print("❌ หาไฟล์น้ำหนักไม่เจอ โปรดตรวจสอบว่ารัน Phase 2 เสร็จสมบูรณ์แล้ว")

model_base.eval()
model_large.eval()

# ── 3. ฟังก์ชันประเมินภาพคู่ (Core Engine) ───────────────────────────
def get_ensemble_score(img_path1, img_path2):
    img1_pil = Image.open(img_path1).convert('RGB')
    img2_pil = Image.open(img_path2).convert('RGB')
    
    i1 = infer_tf(img1_pil).unsqueeze(0).to(device)
    i2 = infer_tf(img2_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            # Base (40%)
            diff_b = (model_base(i2) - model_base(i1) + model_base(torch.flip(i2, [3])) - model_base(torch.flip(i1, [3]))) / 2.0
            prob_b = torch.sigmoid(diff_b).float()
            
            # Large (60%)
            diff_l = (model_large(i2) - model_large(i1) + model_large(torch.flip(i2, [3])) - model_large(torch.flip(i1, [3]))) / 2.0
            prob_l = torch.sigmoid(diff_l).float()

            # รวมร่าง
            final_prob = (prob_b * 0.4) + (prob_l * 0.6)
            
    return final_prob.item()

# ── 4. เริ่มลูปประมวลผล ──────────────────────────────────────────
df = pd.read_csv(CSV_IN)
results = []

print("\nกำลังประมวลผลรูปภาพ...")
for _, row in tqdm(df.iterrows(), total=len(df)):
    p1 = os.path.join(IMG_DIR, row['Image 1'])
    p2 = os.path.join(IMG_DIR, row['Image 2'])

    try:
        # score คือความน่าจะเป็นที่ Image 2 จะชนะ
        score_img2 = get_ensemble_score(p1, p2)
        score_img1 = 1.0 - score_img2
        
        pred_winner = 2 if score_img2 >= 0.5 else 1
        conf        = score_img2 if score_img2 >= 0.5 else score_img1
        
    except Exception as e:
        score_img1 = score_img2 = pred_winner = conf = None
        print(f"\n⚠️ Error อ่านภาพไม่ได้: {row['Image 1']} หรือ {row['Image 2']} -> {e}")

    # ดึงค่า True Winner มาเก็บไว้ (ใช้ .get ป้องกันกรณี CSV ไม่มีคอลัมน์นี้)
    true_winner = row.get('Winner', None)

    results.append({
        'Image 1'     : row['Image 1'],
        'Image 2'     : row['Image 2'],
        'True Winner' : true_winner,
        'score_img1'  : round(score_img1, 4) if score_img1 is not None else None,
        'score_img2'  : round(score_img2, 4) if score_img2 is not None else None,
        'pred_winner' : pred_winner,
        'conf'        : round(conf, 4) if conf is not None else None,
    })

# ── 5. บันทึกผลลัพธ์ ─────────────────────────────────────────────
df_out = pd.DataFrame(results)
df_out.to_csv(CSV_OUT, index=False)
print(f"\n✓ ประมวลผลเสร็จสิ้น! บันทึกไฟล์แล้วที่: {CSV_OUT}")
print(df_out.head())

🚀 เริ่มกระบวนการ Predict Test Set (Weighted Ensemble)
[Init] สถาปัตยกรรม convnext_base_w | Feature Dimension: 640
[Init] สถาปัตยกรรม convnext_large_d | Feature Dimension: 768
✅ โหลดไฟล์น้ำหนัก Phase 2 (Base + Large) สำเร็จ!

กำลังประมวลผลรูปภาพ...


  0%|          | 0/100 [00:00<?, ?it/s]


✓ ประมวลผลเสร็จสิ้น! บันทึกไฟล์แล้วที่: G:\MachineVision\Test Set 1_268\Test Set 1\test_predict.csv
  Image 1 Image 2  True Winner  score_img1  score_img2  pred_winner    conf
0   0.jpg   1.jpg            0      0.1469      0.8531            2  0.8531
1   2.jpg   3.jpg            0      0.6949      0.3051            1  0.6949
2   4.jpg   5.jpg            0      0.0898      0.9102            2  0.9102
3   6.jpg   7.jpg            0      0.9066      0.0934            1  0.9066
4   8.jpg   9.jpg            0      0.1312      0.8688            2  0.8688
